# Milestone 2 lab, notebook 1 of 8: Empty in which rows

## Where we are

| Notebook | What you do |
|---|---|
| **1. Empty in which rows** (you are here) | Write a check for empty cells that does not fire on good rows |
| 2. Unique on which columns | Find the receipt and the file that were counted twice |
| 3. A believable value | Catch 1,180 kg of bananas without catching 400 real bottles |
| 4. Rules of the business | Three rules a person must tell you. Each needs a join first |
| 5. Across two tables | Hold the receipts against the tills' own totals. Find the lines a join dropped |
| 6. A normal day | Find the store that sent nothing. Work out what a normal day is |
| 7. The false alarm | A holiday is not an error. See what silence costs |
| 8. The production version | The same checks as a tested program, and what is still wrong |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`.

**About ten minutes.**

In [1]:
from lab_helpers import keep, lab_copy, on_lab, plant, prepare_week, show_check, sql, standard_checks

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August. 24,384 sales lines.
The five standard checks ran on every night: 35 of 35 passed.


## Step 1: Where you stopped

**Lotus Retails** is a chain of six supermarkets. Customers pay at a **till**: the machine at the counter where a customer pays. It is also called a cash register. Each store manager is paid a bonus of 1% of the store's net sales.

In milestone 1 you found store `S03` reported $18.98 too high, with five green checks. You wrote a check card, and a list of ten candidate checks. Then you stopped. The pipeline is exactly as it was.

Since then the whole bonus week has run: seven nights, 35 checks, all green.

In this milestone you write the checks for `sales_lines`, one kind per notebook. You write each one **by hand first**. Start with the simplest idea there is: a cell must not be empty.

It sounds too easy to need a notebook. By the end of this one you will make a whole store's day disappear, and all five checks will stay green.

## Step 2: The common version

The usual advice is: put a "not empty" check on every column. Try it on one column, `member_id`: the loyalty card of the customer.

In [2]:
sql("""
    SELECT count(*)                                                  AS rows,
           count(*) - count(member_id)                               AS empty_member_id,
           round(100.0 * (count(*) - count(member_id)) / count(*), 1) AS percent_empty
    FROM clean.sales_lines
""")

,rows,empty_member_id,percent_empty
0,24384,15809,64.8


That check would fail on 15,809 rows: 64.8% of the table. Every one of those rows is right. A customer may pay without a loyalty card.

A check that fails on good rows every night teaches people one thing: to ignore it. Your check card already said so in milestone 1. Nobody is hurt by an empty `member_id`, so it gets **no check**.

## Step 3: The other common version

The pipeline has one check for empty cells: standard check 2, "the key is not null". It looks at one column, `line_id`.

Now break something it cannot see. **Samir Haddad** manages store `S05`. Suppose the program in his back office loses the amount column for one night: every `amount_cents` of Wednesday 5 August arrives empty.

You will not break the real table. `lab_copy()` makes a private copy, and `plant()` changes only the copy.

In [3]:
lab_copy()
plant("""
    UPDATE lab.sales_lines SET amount_cents = NULL
    WHERE store_id = 'S05' AND sales_date = DATE '2026-08-05'
""")

lab.sales_lines is a fresh copy of clean.sales_lines: 24,384 rows.
Planted in lab.sales_lines.


Run the five standard checks on the copy.

In [4]:
standard_checks(on_the_lab=True)

1. The table is not empty           0 bad rows  PASS
2. The key is not null              0 bad rows  PASS
3. The key is unique                0 bad rows  PASS
4. The status is an allowed value   0 bad rows  PASS
5. The total is positive            0 bad rows  PASS


Five green checks. Now ask the copy what Samir's store sold that week.

In [5]:
sql(on_lab("""
    SELECT sales_date, count(*) AS lines, sum(amount_cents) AS net_sales_cents
    FROM clean.sales_lines
    WHERE store_id = 'S05' AND sales_date BETWEEN DATE '2026-08-04' AND DATE '2026-08-06'
    GROUP BY sales_date
    ORDER BY sales_date
"""))

,sales_date,lines,net_sales_cents
0,2026-08-04,75,44735.0
1,2026-08-05,82,NaN
2,2026-08-06,88,56427.0


Look at 5 August. 82 lines, and `NaN`: no number at all. The real day was 58804 cents, $588.04.

Two things let it through.

- `sum()` skips empty cells. When every cell is empty, the sum is empty too. It is not zero.
- Check 5 asks "is the total zero or less?". An empty total is not zero or less. So it passes.

Samir's week would be $588.04 short, and his bonus $5.88 short. Nothing would say why.

## Step 4: The careful version, part one

Do not ask "is this column empty?" for every column. Ask: **which columns can a sale never be without?**

A sales line with no amount, no quantity, no price, no time, no product or no receipt is not a sales line. Write that as a query that returns the bad rows.

In [6]:
MUST_BE_FILLED = """
    SELECT line_id, store_id, sales_date, 'a cell that must be filled is empty' AS problem
    FROM clean.sales_lines
    WHERE receipt_id IS NULL OR line_no IS NULL OR sold_at IS NULL OR product_id IS NULL
       OR quantity IS NULL OR unit_price_cents IS NULL OR amount_cents IS NULL
"""
found = sql(on_lab(MUST_BE_FILLED))
print(len(found), "bad rows in the copy")
found.head(3)

82 bad rows in the copy


,line_id,store_id,sales_date,problem
0,S05-0805-000001,S05,2026-08-05,a cell that must be filled is empty
1,S05-0805-000002,S05,2026-08-05,a cell that must be filled is empty
2,S05-0805-000003,S05,2026-08-05,a cell that must be filled is empty


82 rows: every line of Samir's Wednesday. The fault that five checks missed is caught by one `WHERE`.

## Step 5: The careful version, part two

One column is harder. `refund_of_receipt_id` names the receipt that a refund gives money back for. Look at when it is filled.

In [7]:
sql("""
    SELECT line_type, count(*) AS lines, count(refund_of_receipt_id) AS filled
    FROM clean.sales_lines
    GROUP BY line_type
    ORDER BY line_type
""")

,line_type,lines,filled
0,refund,89,89
1,sale,24295,0


It is filled on all 89 refunds, and empty on all 24,295 sales.

So "not empty" is the wrong rule here, and so is "empty". The right rule depends on the row:

- on a **refund**, it must be filled;
- on a **sale**, it must be empty.

Plant one more fault in the copy: a refund that names no receipt. Then write the whole check, with all three parts.

In [8]:
plant("""
    UPDATE lab.sales_lines SET refund_of_receipt_id = NULL
    WHERE line_id = (SELECT min(line_id) FROM lab.sales_lines WHERE line_type = 'refund')
""")

REQUIRED_CELLS = MUST_BE_FILLED + """
    UNION ALL
    SELECT line_id, store_id, sales_date, 'a refund that names no receipt' AS problem
    FROM clean.sales_lines
    WHERE line_type = 'refund' AND refund_of_receipt_id IS NULL
    UNION ALL
    SELECT line_id, store_id, sales_date, 'a sale that names a refunded receipt' AS problem
    FROM clean.sales_lines
    WHERE line_type = 'sale' AND refund_of_receipt_id IS NOT NULL
"""
sql(on_lab(REQUIRED_CELLS))["problem"].value_counts()

Planted in lab.sales_lines.


problem
a cell that must be filled is empty    82
a refund that names no receipt          1
Name: count, dtype: int64

Both planted faults are found, and each row says what is wrong with it.

## Step 6: On the real table

Now run your check on the real table, and hold it against the one in the production code.

In [9]:
yours = sql(REQUIRED_CELLS)
keep("required_cells", yours)

Your query:           0 bad rows
The production check: 0 bad rows  (A cell that must be filled is filled)
The same rows.


Zero rows, and the production check agrees. No store sent an empty cell this week.

A check that finds nothing is not a wasted check. It is there for the night the export program breaks. You have just seen what that night looks like without it.

Here is the production check. It is your query, with its reason written beside it.

In [10]:
show_check("required_cells")

1. A cell that must be filled is filled
   lens:      data
   answers:   Complete?
   protects:  Store managers and the head of retail operations: net sales, so pay
   prepare:   Decide, column by column, in which rows an empty cell is wrong
   limit:     No empty amount, quantity, price, time, product or receipt. A refund names its receipt, and a sale does not

        SELECT line_id, store_id, sales_date, 'a cell that must be filled is empty' AS problem
        FROM clean.sales_lines
        WHERE receipt_id IS NULL OR line_no IS NULL OR sold_at IS NULL OR product_id IS NULL
           OR quantity IS NULL OR unit_price_cents IS NULL OR amount_cents IS NULL
        UNION ALL
        SELECT line_id, store_id, sales_date, 'a refund that names no receipt' AS problem
        FROM clean.sales_lines
        WHERE line_type = 'refund' AND refund_of_receipt_id IS NULL
        UNION ALL
        SELECT line_id, store_id, sales_date, 'a sale that names a refunded receipt' AS problem
        FROM

This kind of check has a name: a **not-null check**. It checks that a cell is filled. ("Null" is the database's word for an empty cell.)

Writing `IS NULL` is easy. The skill is saying **in which rows** an empty cell is wrong.

| Version | On good data | On the planted faults |
|---|---|---|
| "Not null" on every column | Fails on 15,809 good rows | |
| "Not null" on the key only | Passes | Passes. A store's day disappears |
| Filled where it must be, row by row | Passes | Fails, and says why |

## What you now know

- An empty cell is not wrong by itself. Ask who is hurt when it is empty.
- `sum()` skips empty cells. A day with every amount empty has no total, and "total above zero" still passes.
- The same column can need to be filled in one kind of row and empty in another.
- A not-null check is a check that a cell is filled.

**Say it in an interview:** "I do not put a not-null check on every column. On the sales lines, an empty loyalty id was normal in 65% of rows, so it got none. An empty amount got one, because a sum skips empty cells: a store's whole day vanished and our 'total is positive' check stayed green."

**The theory:** [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), section "A starter kit for any table".

Next: [notebook 2](02_unique_on_which_columns.ipynb). One store's Friday is twice its normal size, and the "key is unique" check is green.